In [1]:
import re
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Any
from datasets import load_dataset
from tqdm.auto import tqdm

e:\work\LLM\RAG\example\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------

# https://huggingface.co/datasets/codersan/Persian-Wikipedia-Corpus?utm_source=chatgpt.com


from pathlib import Path
from datasets import load_dataset, Dataset, load_from_disk

# --------------------------------------------------
# 1. Settings
# --------------------------------------------------

N = 10_000
DATA_DIR = Path("data/persian_wikipedia_10k")

# --------------------------------------------------
# 2. Load from local disk or download
# --------------------------------------------------

if DATA_DIR.exists():

    print("Loading dataset from local disk...")

    dataset = load_from_disk(str(DATA_DIR))

else:

    print("Streaming dataset from Hugging Face...")

    streamed_dataset = load_dataset(
        "codersan/Persian-Wikipedia-Corpus",
        split="train",
        streaming=True
    )

    # Get only the first 10,000 records
    rows = list(streamed_dataset.take(N))

    print(f"Received {len(rows)} records.")
    print("Saving dataset locally...")

    dataset = Dataset.from_list(rows)

    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(DATA_DIR))

    print("Dataset saved successfully!")

# --------------------------------------------------
# 3. Convert to Pandas
# --------------------------------------------------

df = dataset.to_pandas()

print("Dataset loaded successfully!")
print("Number of samples:", len(df))
print(df.head())


Loading dataset from local disk...
Dataset loaded successfully!
Number of samples: 10000
   Id        Title  Type  Rank  Namespace  \
0   1   صفحهٔ اصلی     0    38          0   
1  42    ویکی‌پدیا     6   728          0   
2  47      سالنامه     0    29          0   
3  49      اطلاعات     0   211          0   
4  52  محتوای آزاد     0   129          0   

                                        RedirectList  IsDisambiguationPage  \
0  [صفحه‌ی اصلی, صفحۀ اصلی, صفحه اصلی, صفحهٔ اصلي...                 False   
1  [ویکی پدیا, ویکیپدیا, دانشنامهٔ ویکیپدیا, دانش...                 False   
2                                                 []                 False   
3                                                 []                 False   
4  [محتويات آزاد, محتویات آزاد, محتواي آزاد, محتو...                 False   

   TargetLinksCount                                            InfoBox  \
0                30                 {'KeysAndValues': [], 'Title': ''}   
1               313  {'

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Id                    10000 non-null  int64 
 1   Title                 10000 non-null  str   
 2   Type                  10000 non-null  int64 
 3   Rank                  10000 non-null  int64 
 4   Namespace             10000 non-null  int64 
 5   RedirectList          10000 non-null  object
 6   IsDisambiguationPage  10000 non-null  bool  
 7   TargetLinksCount      10000 non-null  int64 
 8   InfoBox               10000 non-null  object
 9   Text                  10000 non-null  str   
 10  Links                 10000 non-null  object
 11  Parents               10000 non-null  object
dtypes: bool(1), int64(5), object(4), str(2)
memory usage: 73.9+ MB


In [4]:
# print(df.head)
print(df.iloc[1].to_dict())
print(f"Initial documents: {len(df)}")

{'Id': 42, 'Title': 'ویکی\u200cپدیا', 'Type': 6, 'Rank': 728, 'Namespace': 0, 'RedirectList': array(['ویکی پدیا', 'ویکیپدیا', 'دانشنامهٔ ویکیپدیا', 'دانشنامه ویکیپدیا',
       'دانشنامه جهان\u200cشمول ویکی\u200cپدیا', 'Wikipedia',
       'دانشنامه ویکی\u200cپدیا', 'دانشنامه ویکی پدیا', 'ويكي پديا',
       'ويكيپديا', 'ویکی\u200cپدیا، دانشنامهٔ آزاد',
       'دانشنامهٔ ویکی\u200cپدیا', 'دانشنامه جهان شمول ويكي پديا',
       'دانشنامه جهان شمول ویکی پدیا',
       'دانشنامهٔ جهان\u200cشمول ویکی\u200cپدیا', 'دانشنامه ويكي پديا',
       'دانشنامهٔ ویکی پدیا', 'دانشنامه ويكيپديا', 'دانشنامهٔ ويكيپديا',
       'دانشنامهٔ ويكي پديا', 'دانشنامهٔ جهان شمول ويكي پديا',
       'دانشنامهٔ جهان شمول ویکی پدیا', 'ويكي پديا، دانشنامهٔ آزاد',
       'ویکی پدیا، دانشنامهٔ آزاد', 'ویکی\u200cپدیا، دانشنامهٔ ازاد',
       'ويكي پديا، دانشنامهٔ ازاد', 'ویکی پدیا، دانشنامهٔ ازاد',
       'تو ویکیپدیا ویرایش کردن'], dtype=object), 'IsDisambiguationPage': False, 'TargetLinksCount': 313, 'InfoBox': {'KeysAndVal

In [5]:
# --------------------------------------------------
# 2. Basic filtering
# --------------------------------------------------

# Main Wikipedia namespace only
df = df[df["Namespace"] == 0].copy()

# Remove disambiguation pages
df = df[
    ~df["IsDisambiguationPage"]
].copy()

# Remove missing title/text
df = df[
    df["Title"].notna() &
    df["Text"].notna()
].copy()

print(f"After structural filtering: {len(df)}")

After structural filtering: 9228


In [6]:
# --------------------------------------------------
# 3. Persian normalization
# --------------------------------------------------

def normalize_persian_text(text):

    if not isinstance(text, str):
        return ""

    # Arabic Yeh variants
    text = text.replace("ي", "ی")
    text = text.replace("ى", "ی")

    # Arabic Kaf
    text = text.replace("ك", "ک")

    # Remove BOM / zero-width space
    text = text.replace("\ufeff", "")
    text = text.replace("\u200b", "")

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [7]:
tqdm.pandas()

df["clean_title"] = df["Title"].progress_apply(
    normalize_persian_text
)

df["clean_text"] = df["Text"].progress_apply(
    normalize_persian_text
)

100%|██████████| 9228/9228 [00:04<00:00, 2047.50it/s]


In [8]:
MIN_CHARS = 200

df = df[
    df["clean_text"].str.len() >= MIN_CHARS
].copy()

print(f"After length filtering: {len(df)}")

After length filtering: 8624


In [9]:
before = len(df)

df = df.drop_duplicates(
    subset=["clean_text"]
).copy()

print(
    f"Removed duplicates: {before - len(df)}"
)

Removed duplicates: 0


In [10]:
TYPE_MAP = {
    0: "Unknown",
    1: "Person",
    2: "Location",
    3: "Organization",
    4: "Event",
    6: "Other"
}

In [11]:
@dataclass
class Document:
    page_content: str
    metadata: Dict[str, Any]

In [12]:
def row_to_document(row):

    title = row["clean_title"]
    text = row["clean_text"]

    content = (
        f"عنوان: {title}\n\n"
        f"{text}"
    )

    metadata = {
        "doc_id": int(row["Id"]),
        "title": title,
        "type": int(row["Type"]),
        "type_name": TYPE_MAP.get(
            int(row["Type"]),
            "Unknown"
        ),
        "rank": int(row["Rank"]),
        "categories": row["Parents"],
    }

    return Document(
        page_content=content,
        metadata=metadata
    )

In [13]:
documents = [
    row_to_document(row)
    for _, row in tqdm(
        df.iterrows(),
        total=len(df)
    )
]

print("Number of documents:", len(documents))

100%|██████████| 8624/8624 [00:01<00:00, 7622.13it/s] 

Number of documents: 8624


In [14]:
doc = documents[10]

print("CONTENT")
print("=" * 80)
print(doc.page_content[:1000])

print("\nMETADATA")
print("=" * 80)
print(doc.metadata)

CONTENT
عنوان: گاه‌شماری میلادی

جشن‌گرفتن گاه‌شماری گریگوری نزد پاپ گریگوری سیزدهم، اثر کامیلو روسکنی (تکمیل‌شده در سال ۱۷۲۳)
گاه‌شماری میلادی گاه‌شماری با ریشه مسیحی است که هم‌اکنون در بیشتر کشورهای جهان استفاده می‌شود و تقویمی بین‌المللی است. این گاه‌شماری برگرفته از گاه‌شماری ژولینی با مبدا میلادی (میلاد عیسی) است و نخست از سوی آلویسیوس لیلیوس که پزشکی از اهالی ناپل بود پیشنهاد شد. تعدیل کبیسه‌گیری این گاهشماری در ۲۴ فوریه ۱۵۸۲ از سوی پاپ گریگوری سیزدهم پذیرفته شد و از آن پس به گاه‌شماری گریگوری (گرگوری) نیز مشهور شد.
تاریخچه و ویژگی‌ها
گاه‌شماری ژولینی که از گاه‌شماری رومی گرفته شده بود در دوره مسیحی با مبدا میلادی (گاه‌شماری میلادی قدیم) پذیرفته شد. این گاه‌شماری ۳۶۵ روز بود که برای هر ۴ سال یک روز به روزهای سال اضافه می‌شد (سال ۳۶۵٫۲۵ روزه). به علت دقیق نبودن کسر سال نسبت به سال اعتدالی هر ۱۲۸ سال یک روز عقب می‌افتاد. در سال ۱۵۸۲ میلادی جهت تصحیح آن گاه‌شماری گرگوری (میلادی جدید) ایجاد شد. در گاه‌شماری یولیانی اعتدال بهاری در سال ۳۲۵ میلادی که شورای نیقیه تشکیل شده بود در ۲۱ مار

# Chunking

In [15]:

import re
from typing import List, Dict, Any

# --------------------------------------------------
# 1. Chunking configuration
# --------------------------------------------------

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200


# --------------------------------------------------
# 2. Split long paragraphs
# --------------------------------------------------

def split_long_text(text, chunk_size, overlap):

    chunks = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        # Try to avoid cutting words
        if end < len(text):
            boundary = text.rfind(" ", start, end)

            if boundary > start:
                end = boundary

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return chunks


# --------------------------------------------------
# 3. Document chunking
# --------------------------------------------------

def chunk_document(document):

    text = document.page_content

    # Split using paragraphs
    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(paragraph) > CHUNK_SIZE:

            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            chunks.extend(
                split_long_text(
                    paragraph,
                    CHUNK_SIZE,
                    CHUNK_OVERLAP
                )
            )

            continue

        candidate = (
            current_chunk + "\n\n" + paragraph
            if current_chunk
            else paragraph
        )

        if len(candidate) <= CHUNK_SIZE:

            current_chunk = candidate

        else:

            if current_chunk:
                chunks.append(current_chunk)

            current_chunk = paragraph

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


# --------------------------------------------------
# 4. Convert all documents to chunks
# --------------------------------------------------

chunk_records = []

for document in documents:

    chunks = chunk_document(document)

    for chunk_index, chunk_text in enumerate(chunks):

        chunk_records.append({

            "text": chunk_text,

            "metadata": {
                **document.metadata,
                "chunk_index": chunk_index,
            }

        })


print("Original documents:", len(documents))
print("Total chunks:", len(chunk_records))

print("\nExample chunk:")
print(chunk_records[0]["text"][:500])

print("\nExample metadata:")
print(chunk_records[0]["metadata"])


Original documents: 8624
Total chunks: 51958

Example chunk:
عنوان: صفحهٔ اصلی

پیوند= مقاله‌های برگزیده – مقاله پیشنهادی هفته
درباره ویکی‌پدیا
درون‌مایه
از میان خبرها
امروز : ، میلادی برابر هجری خورشیدی و (UTC)
-1 day – +1 day یادبودهای – یادبودهای بیشتر…
link= بایگانی – نگاره‌های برگزیده بیشتر
دیگر پروژه‌های بنیاد ویکی‌مدیا
ویکی‌پدیا در زبان‌های دیگر

Example metadata:
{'doc_id': 1, 'title': 'صفحهٔ اصلی', 'type': 0, 'type_name': 'Unknown', 'rank': 38, 'categories': array([], dtype=object), 'chunk_index': 0}


In [16]:
from sentence_transformers import SentenceTransformer

# Load model
MODEL_NAME = "intfloat/multilingual-e5-small"

model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

# Test documents
documents = [
    "passage: تهران پایتخت ایران است.",
    "passage: اصفهان یکی از شهرهای تاریخی ایران است.",
    "passage: Machine learning is a branch of artificial intelligence."
]

# Generate embeddings
embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    batch_size=3
)

print("Embedding shape:", embeddings.shape)
print("Vector dimension:", embeddings.shape[1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1141.06it/s]


Embedding shape: (3, 384)
Vector dimension: 384


In [ ]:

# import torch
# from FlagEmbedding import BGEM3FlagModel

# # --------------------------------------------------
# # 1. Load model
# # --------------------------------------------------

# MODEL_NAME = "BAAI/bge-m3"

# device = "cuda" if torch.cuda.is_available() else "cpu"

# print("Device:", device)

# model = BGEM3FlagModel(
#     MODEL_NAME,
#     use_fp16=(device == "cuda")
# )

# # --------------------------------------------------
# # 2. Test embedding
# # --------------------------------------------------

# test_texts = [
#     "تهران پایتخت ایران است.",
#     "اصفهان یکی از شهرهای تاریخی ایران است.",
#     "Machine learning is a branch of artificial intelligence."
# ]

# test_output = model.encode(
#     test_texts,
#     batch_size=3,
#     max_length=512
# )

# test_vectors = test_output["dense_vecs"]

# print("Embedding shape:", test_vectors.shape)
# print("Vector dimension:", test_vectors.shape[1])


# Device: cpu
# Could not reach the Hub ([WinError 10061] No connection could be made because the target machine actively refused it). Using cached commit hash for 'BAAI/bge-m3'.
# '[WinError 10061] No connection could be made because the target machine actively refused it' thrown while requesting GET https://huggingface.co/api/models/BAAI/bge-m3/tree/main/additional_chat_templates
# Retrying in 1s [Retry 1/5].
# Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]


Device: cpu


Could not reach the Hub ([WinError 10061] No connection could be made because the target machine actively refused it). Using cached commit hash for 'BAAI/bge-m3'.
'[WinError 10061] No connection could be made because the target machine actively refused it' thrown while requesting GET https://huggingface.co/api/models/BAAI/bge-m3/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:

from pathlib import Path

from qdrant_client import QdrantClient, models

# --------------------------------------------------
# 1. Database configuration
# --------------------------------------------------

QDRANT_PATH = "data/qdrant_db"

COLLECTION_NAME = "persian_wikipedia"

VECTOR_SIZE = 1024

# --------------------------------------------------
# 2. Initialize Qdrant
# --------------------------------------------------

client = QdrantClient(
    path=QDRANT_PATH
)

# --------------------------------------------------
# 3. Create collection if it doesn't exist
# --------------------------------------------------

if not client.collection_exists(COLLECTION_NAME):

    client.create_collection(

        collection_name=COLLECTION_NAME,

        vectors_config=models.VectorParams(
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE
        )

    )

    print("Collection created!")

else:

    print("Collection already exists!")

print(client.get_collection(COLLECTION_NAME))


In [ ]:

import uuid
import numpy as np
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Indexing configuration
# --------------------------------------------------

BATCH_SIZE = 16

# --------------------------------------------------
# 2. Process and upload chunks
# --------------------------------------------------

for start in tqdm(
    range(0, len(chunk_records), BATCH_SIZE)
):

    batch = chunk_records[
        start:start + BATCH_SIZE
    ]

    texts = [
        item["text"]
        for item in batch
    ]

    # Generate embeddings
    output = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        max_length=512
    )

    vectors = output["dense_vecs"]

    points = []

    for i, item in enumerate(batch):

        metadata = item["metadata"]

        # Deterministic ID:
        # Same document + same chunk = same point ID

        point_id = str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"wiki:{metadata['doc_id']}:{metadata['chunk_index']}"
            )
        )

        payload = {
            "text": item["text"],
            **metadata
        }

        points.append(
            models.PointStruct(
                id=point_id,
                vector=vectors[i].tolist(),
                payload=payload
            )
        )

    # Upload batch
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True
    )

print("Indexing completed!")

print(
    "Total points:",
    client.count(
        collection_name=COLLECTION_NAME,
        exact=True
    ).count
)


In [ ]:

# --------------------------------------------------
# 1. Semantic search function
# --------------------------------------------------

def semantic_search(query, top_k=5):

    # Convert query to embedding
    output = model.encode(
        [query],
        batch_size=1,
        max_length=512
    )

    query_vector = output["dense_vecs"][0].tolist()

    # Search in Qdrant
    results = client.query_points(

        collection_name=COLLECTION_NAME,

        query=query_vector,

        limit=top_k,

        with_payload=True

    ).points

    # Display results
    retrieved_documents = []

    for result in results:

        item = {
            "score": result.score,
            "text": result.payload["text"],
            "title": result.payload["title"],
            "doc_id": result.payload["doc_id"],
            "chunk_index": result.payload["chunk_index"]
        }

        retrieved_documents.append(item)

    return retrieved_documents


# --------------------------------------------------
# 2. Test retrieval
# --------------------------------------------------

query = "تاریخچه شکل گیری دانشگاه های ایران"

results = semantic_search(
    query,
    top_k=5
)

for i, result in enumerate(results):

    print("=" * 80)

    print(f"Result {i+1}")
    print("Title:", result["title"])
    print("Score:", round(result["score"], 4))
    print("Document ID:", result["doc_id"])

    print("\nContent:")
    print(result["text"][:700])
